# 02. Baseline Models & Comparison

**Сравнение разных подходов к рекомендациям**

Цель: обосновать выбор Sentence-Transformers + Vector Search как основного решения.

In [11]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import seaborn as sns
from sentence_transformers import SentenceTransformer
import time
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8')
%matplotlib inline

In [12]:
df = pd.read_csv("high_popularity_spotify_data.csv")

# Создаём текстовое представление
def create_text_representation(row):
    return f"{row.get('track_name', '')} by {row.get('track_artist', '')} " \
           f"genre {row.get('playlist_genre', '')} " \
           f"mood {row.get('mood', 'neutral')}"

df['text'] = df.apply(create_text_representation, axis=1)
print(f"Dataset loaded: {len(df)} tracks")

Dataset loaded: 1686 tracks


In [13]:
print("=== TF-IDF + Cosine Similarity ===")
start = time.time()

tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['text'])

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
print(f"Time: {time.time() - start:.2f} sec")

=== TF-IDF + Cosine Similarity ===
TF-IDF matrix shape: (1686, 3093)
Time: 0.05 sec


In [14]:
print("=== Sentence-Transformers ===")
start = time.time()

model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(df['text'].tolist(), show_progress_bar=True, normalize_embeddings=True)

print(f"Embeddings shape: {embeddings.shape}")
print(f"Time: {time.time() - start:.2f} sec")

=== Sentence-Transformers ===


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/53 [00:00<?, ?it/s]

Embeddings shape: (1686, 384)
Time: 25.32 sec


In [15]:
def get_recommendations(query: str, top_k=5, method='st'):
    if method == 'tfidf':
        query_vec = tfidf.transform([query])
        sims = cosine_similarity(query_vec, tfidf_matrix).flatten()
    else:  # sentence-transformers
        query_vec = model.encode([query], normalize_embeddings=True)
        sims = (query_vec @ embeddings.T).flatten()

    top_indices = sims.argsort()[-top_k:][::-1]
    results = df.iloc[top_indices][['track_name', 'track_artist', 'playlist_genre']].copy()
    results['similarity'] = sims[top_indices]
    return results

# Тестовые запросы
test_queries = [
    "energetic workout music",
    "chill rainy day lo-fi",
    "romantic evening dinner",
    "focus coding session",
    "happy summer party"
]

In [16]:
print("=== Сравнение моделей ===\n")

for query in test_queries:
    print(f"\nQuery: '{query}'")
    print("-" * 60)

    print("TF-IDF:")
    print(get_recommendations(query, method='tfidf')[['track_name', 'track_artist', 'similarity']].head(3))

    print("\nSentence-Transformers:")
    print(get_recommendations(query, method='st')[['track_name', 'track_artist', 'similarity']].head(3))
    print("="*80)

=== Сравнение моделей ===


Query: 'energetic workout music'
------------------------------------------------------------
TF-IDF:
                                 track_name  \
938                    Don't Stop The Music   
1409                                  Porno   
744   Quevedo: Bzrp Music Sessions, Vol. 52   

                                           track_artist  similarity  
938                                             Rihanna    0.536037  
1409  Rich Music LTD, Sech, Dalex, Justin Quiles, Le...    0.311910  
744                                   Bizarrap, Quevedo    0.284659  

Sentence-Transformers:
                             track_name                   track_artist  \
931   Stereo Hearts (feat. Adam Levine)  Gym Class Heroes, Adam Levine   
1485                           Sprinter              Dave, Central Cee   
163                                  25                       Rod Wave   

      similarity  
931     0.634662  
1485    0.599584  
163     0.563699  

Que

## Выводы по сравнению моделей

**Преимущества Sentence-Transformers:**
- Лучшее понимание семантики
- Работает с синонимами и контекстом
- Более релевантные рекомендации

In [17]:
# Сравнение времени инференса
queries = ["test query"] * 50

start = time.time()
for q in queries:
    get_recommendations(q, method='tfidf', top_k=10)
tfidf_time = time.time() - start

start = time.time()
for q in queries:
    get_recommendations(q, method='st', top_k=10)
st_time = time.time() - start

print(f"TF-IDF time: {tfidf_time:.2f}s")
print(f"ST time:     {st_time:.2f}s")
print(f"ST быстрее в {tfidf_time/st_time:.1f} раз")

TF-IDF time: 0.15s
ST time:     0.98s
ST быстрее в 0.2 раз
